<a href="https://colab.research.google.com/github/mdrs-thiago/mdrs-thiago.github.io/blob/main/topicos_a/2027/notebooks/lab06_rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🔬 Lab 6 — Sistema RAG
**Tópicos Especiais A · Deep Learning · UERJ 2027.1** · Aula 24 · Bloco 6 — Modelos Generativos e LLMs

**Objetivo:** construir um sistema RAG com modelos abertos e avaliá-lo com 10 perguntas de resposta conhecida.

| | |
|---|---|
| ✅ **Checkpoint (em sala)** | 60 XP: veja as marcações `✅ CHECKPOINT` |
| 📦 **Entrega (7 dias)** | 180 XP: pipeline completo, tabela de avaliação, comparação com/sem RAG, 3 falhas analisadas e 1 experimento de melhoria. |

> Preencha apenas os trechos marcados com `# TODO`. Antes de começar: **Arquivo → Salvar uma cópia no Drive** e, se indicado, **Ambiente de execução → Alterar tipo → T4 GPU**.

In [ ]:
!pip -q install pypdf sentence-transformers faiss-cpu transformers accelerate

In [ ]:
import random, numpy as np, torch
def set_seed(s=42):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device, '| torch', torch.__version__)

## 1. Corpus
Sugestão: o **regulamento de graduação da UERJ** (PDF público no site da universidade) ou um corpus seu de 20–200 páginas.
Faça upload do PDF no painel de arquivos.

In [ ]:
from pypdf import PdfReader
PDF = 'documento.pdf'   # TODO: nome do seu arquivo
pages = [(i + 1, p.extract_text() or '') for i, p in enumerate(PdfReader(PDF).pages)]
print(len(pages), 'páginas'); print(pages[0][1][:500])

## 2. Chunking com sobreposição (guardando a página de origem)

In [ ]:
import re
def chunk_page(text, size=900, overlap=150):
    text = re.sub(r'\s+', ' ', text).strip()
    # TODO: divida o texto em pedaços de `size` caracteres com `overlap` de sobreposição
    return ...

chunks = [{'page': pg, 'text': c} for pg, t in pages for c in chunk_page(t) if len(c) > 50]
print(len(chunks), 'chunks'); chunks[3]

## 3. Embeddings e índice FAISS

In [ ]:
from sentence_transformers import SentenceTransformer
import faiss
emb = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2', device=device)
X = emb.encode([c['text'] for c in chunks], normalize_embeddings=True, show_progress_bar=True).astype('float32')
index = faiss.IndexFlatIP(X.shape[1]); index.add(X)

def retrieve(q, k=4):
    qv = emb.encode([q], normalize_embeddings=True).astype('float32')
    scores, ids = index.search(qv, k)
    return [{**chunks[i], 'score': float(s)} for s, i in zip(scores[0], ids[0])]

## 4. ✅ CHECKPOINT
1. Teste `retrieve()` com **3 perguntas** e confira se os trechos fazem sentido.
2. Escreva as **10 perguntas de avaliação** (seção 6) **antes** de testar o gerador, incluindo 2 cuja resposta **não** está no documento.

In [ ]:
for q in ['TODO pergunta 1', 'TODO pergunta 2', 'TODO pergunta 3']:
    print('❓', q)
    for h in retrieve(q, 3): print(f"  [p. {h['page']}] ({h['score']:.2f}) {h['text'][:150]}...")

## 5. Gerador

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
LLM = 'Qwen/Qwen2.5-1.5B-Instruct'
tok = AutoTokenizer.from_pretrained(LLM)
llm = AutoModelForCausalLM.from_pretrained(LLM, torch_dtype='auto', device_map='auto')

def generate(msgs, temperature=0.1, max_new_tokens=300):
    ids = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors='pt').to(llm.device)
    out = llm.generate(ids, max_new_tokens=max_new_tokens, do_sample=temperature > 0,
                       temperature=max(temperature, 1e-5), top_p=0.9)
    return tok.decode(out[0, ids.shape[1]:], skip_special_tokens=True)

SYSTEM = ("Você responde perguntas sobre o documento fornecido. Use APENAS os trechos abaixo. "
          "Cite a página entre colchetes, ex.: [p. 12]. Se a resposta não estiver nos trechos, "
          "diga 'Não encontrei no documento'.")

def answer(q, k=4):
    hits = retrieve(q, k)
    ctx = '\n\n'.join(f"[p. {h['page']}] {h['text']}" for h in hits)
    # TODO: monte as mensagens (system + user com <trechos>...</trechos> e a pergunta) e chame generate
    msgs = ...
    return generate(msgs), hits

def answer_no_rag(q):
    return generate([{'role': 'user', 'content': q}])

## 6. Avaliação sistemática

In [ ]:
import pandas as pd
# TODO: 10 perguntas com resposta e página esperadas (2 delas SEM resposta no documento: page=None)
evalset = [
    {'q': '...', 'esperado': '...', 'page': 1},
]
rows = []
for item in evalset:
    resp, hits = answer(item['q'])
    rows.append({'pergunta': item['q'], 'esperado': item['esperado'], 'página': item['page'],
                 'retrieval ok': item['page'] in [h['page'] for h in hits] if item['page'] else None,
                 'resposta RAG': resp, 'resposta sem RAG': answer_no_rag(item['q'])})
df = pd.DataFrame(rows)
df.to_csv('avaliacao_rag.csv', index=False); df

Agora **anote manualmente** no CSV (ou numa cópia da tabela) para cada pergunta: resposta correta? (✅/⚠️/❌) · citou a página certa? · sem RAG: correta?
Calcule: recall@k do retrieval, % de respostas corretas, % de citações corretas, % de "não encontrei" nas 2 perguntas sem resposta.

## 7. Um experimento de melhoria
Escolha **um**: tamanho do chunk, valor de k, busca híbrida (BM25 + vetorial) ou reranker (cross-encoder). Mostre antes/depois nas mesmas 10 perguntas.

In [ ]:
# TODO: seu experimento

## 8. Análise (📦 entrega)
- Tabela de avaliação e métricas agregadas.
- Com RAG × sem RAG: o que mudou?
- **3 falhas**: foi de retrieval ou de geração? Como você corrigiria?
- Resultado do experimento de melhoria.